# 17 · 用 CNN 识别 MNIST

> **本节属于 Part 6 · 卷积神经网络 (CNN)。这是 Part 6 的收尾实战。**

积木已备齐：`Conv2d`、`MaxPool2d`、`Flatten`、`ReLU`、`Linear`。本节我们把它们搭成一个 **LeNet 风格的 CNN**，训练它识别 MNIST——准确率将**超过**我们之前的 MLP——并可视化它**自己学到**的卷积核与特征图。

## 学习目标

- 用 `minitorch.nn` 搭建并训练一个 CNN，准确率超过 MLP（95.7%）
- **可视化**网络第一层学到的卷积核与特征图
- 与 PyTorch 的等价 CNN 对照（看纯 CPU 框架与成熟框架的差距与一致性）

## 搭建 CNN 并加载数据

结构：`Conv(1→16) → ReLU → Pool → Conv(16→32) → ReLU → Pool → Flatten → Linear → 10`。注意输入要整形成 `(N, 1, 28, 28)`（批量、通道、高、宽）。

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import minitorch
from minitorch import Tensor, nn, no_grad, data
from minitorch.optim import Adam

(X_tr, y_tr), (X_te, y_te) = minitorch.utils.load_mnist(n_train=12000, n_test=2000, flatten=False)
X_tr = X_tr.reshape(-1, 1, 28, 28); X_te = X_te.reshape(-1, 1, 28, 28)
print("数据形状:", X_tr.shape)

def make_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28->14
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14->7
        nn.Flatten(), nn.Linear(32 * 7 * 7, 10),
    )

minitorch.set_seed(0)
model = make_cnn()
print("参数总量:", f"{sum(p.data.size for p in model.parameters()):,}")

## 训练

> 提示：纯 NumPy(float64) 的 CNN 在 CPU 上较慢，下面约需一两分钟。我们追求的是**理解原理**；极致速度交给 PyTorch 那一半。

In [ ]:
opt = Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
loader = data.DataLoader(data.TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)

def accuracy():
    model.eval()
    with no_grad():
        return (model(Tensor(X_te)).data.argmax(1) == y_te).mean()

accs = []
t0 = time.time()
for ep in range(7):
    model.train()
    for xb, yb in loader:
        opt.zero_grad()
        loss_fn(model(Tensor(xb)), yb).backward()
        opt.step()
    accs.append(accuracy())
    print(f"epoch {ep}  test_acc {accs[-1]*100:.2f}%  ({time.time()-t0:.0f}s)")

best = max(accs)
n_params = sum(p.data.size for p in model.parameters())
print(f"\nCNN 最佳准确率 {best*100:.2f}%   （对比 nb10 的 MLP：95.7%，且 MLP 有 ~101,770 参数）")
print(f"CNN 仅用 {n_params:,} 个参数 —— 约 MLP 的 1/5，却达到相当甚至更高的准确率！")

## 可视化：网络学到了什么

### 第一层卷积核
第一层的 16 个 3×3 卷积核——它们是网络**自己学出来**的特征检测器（类似边缘/角点）。

In [ ]:
W = model.layers[0].weight.data    # (16, 1, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(10, 2.6))
for i, ax in enumerate(axes.flat):
    ax.imshow(W[i, 0], cmap="gray"); ax.axis("off")
plt.suptitle("Learned conv1 filters (3x3)"); plt.tight_layout(); plt.show()

### 特征图
把一张数字喂进第一层卷积+ReLU，看 8 个通道的响应。

In [ ]:
conv1 = nn.Sequential(model.layers[0], model.layers[1])   # Conv + ReLU
with no_grad():
    fmaps = conv1(Tensor(X_te[0:1])).data[0]              # (16, 28, 28)
fig, axes = plt.subplots(1, 9, figsize=(12, 1.6))
axes[0].imshow(X_te[0, 0], cmap="gray"); axes[0].set_title("input"); axes[0].axis("off")
for i in range(8):
    axes[i+1].imshow(fmaps[i], cmap="gray"); axes[i+1].set_title(f"ch{i}"); axes[i+1].axis("off")
plt.tight_layout(); plt.show()

### 预测示例

In [ ]:
with no_grad():
    pred = model(Tensor(X_te[:10])).data.argmax(1)
fig, axes = plt.subplots(1, 10, figsize=(13, 1.6))
for i, ax in enumerate(axes):
    ax.imshow(X_te[i, 0], cmap="gray")
    ax.set_title(str(pred[i]), color="green" if pred[i] == y_te[i] else "red"); ax.axis("off")
plt.suptitle("CNN predictions"); plt.tight_layout(); plt.show()

## PyTorch 对照

同样结构的 PyTorch CNN（在更多数据上多训几轮，纯 CPU 也很快），可以达到 ~99%——这就是我们 minitorch 实现的"正确性 + 可训练性"在成熟框架上的天花板。

In [ ]:
import torch
import torch.nn as tnn

(Xtr2, ytr2), _ = minitorch.utils.load_mnist(n_train=20000, n_test=2000, flatten=False)
Xtr2 = torch.tensor(Xtr2.reshape(-1, 1, 28, 28), dtype=torch.float32); ytr2 = torch.tensor(ytr2)
Xte_t = torch.tensor(X_te, dtype=torch.float32)

torch.manual_seed(0)
tmodel = tnn.Sequential(
    tnn.Conv2d(1, 16, 3, padding=1), tnn.ReLU(), tnn.MaxPool2d(2),
    tnn.Conv2d(16, 32, 3, padding=1), tnn.ReLU(), tnn.MaxPool2d(2),
    tnn.Flatten(), tnn.Linear(32 * 7 * 7, 10))
topt = torch.optim.Adam(tmodel.parameters(), lr=1e-3); tlf = tnn.CrossEntropyLoss()
for ep in range(4):
    perm = torch.randperm(len(Xtr2))
    for i in range(0, len(Xtr2), 128):
        b = perm[i:i+128]
        topt.zero_grad(); tlf(tmodel(Xtr2[b]), ytr2[b]).backward(); topt.step()
with torch.no_grad():
    tacc = (tmodel(Xte_t).argmax(1).numpy() == y_te).mean()
print(f"PyTorch CNN 准确率: {tacc*100:.2f}%   |   minitorch CNN: {accs[-1]*100:.2f}%")

## 📦 沉淀进 minitorch

本节没有新增框架代码——我们用 Part 6 造好的 `Conv2d/MaxPool2d/Flatten` 搭出了完整 CNN。**CNN 在图像上胜过 MLP**，正是因为它利用了局部连接与参数共享这两个图像先验。

## 小练习

1. **更深的 CNN**：再加一组 `Conv+ReLU+Pool`，或把通道数加倍，准确率/耗时如何变化？
2. **去掉卷积**：把卷积层换成同参数量的全连接，准确率会下降吗？（体会卷积的归纳偏置。）
3. **看错样本**：可视化被分错的数字，它们是否确实难辨认？

## 小结 & 下一站

✅ 我们用 minitorch 搭建并训练了 CNN，**用约 1/5 的参数**达到了与 MLP 相当甚至更高的准确率（这正是卷积"权重共享"的威力），并可视化了它自动学到的卷积核与特征图。**Part 6 完成！**

**下一站 → Part 7 `18_rnn_cell_and_bptt`**：进入**序列建模**。我们将实现循环神经网络 RNN，理解"随时间反向传播 (BPTT)"——并惊喜地发现：我们的 autograd 引擎能**自动**完成 BPTT，无需任何额外推导。